# E1: Fit the fingerprints (exploration notebook)

Three parts, one step per cell:

1. **Build chroma** from the audio clips (the 12 numbers the classifier sees).
2. **Test the fitting code on a fake instrument**, where we know the right answer.
3. **Fit a fingerprint *h* for each real instrument** and see how well it explains its chroma (R²).

Run it from the `src/` folder (that's where it lives), top to bottom.

In [ ]:
import os, json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import nnls

DATA = "data/processed"          
SR = 16000                       
PITCH_CLASSES = ["C","C#","D","Eb","E","F","F#","G","Ab","A","Bb","B"]        
QUALITIES = {"maj":[0,4,7], "min":[0,3,7], "maj7":[0,4,7,11], "min7":[0,3,7,10],
             "dom7":[0,4,7,10], "m7b5":[0,3,6,10], "power":[0,7]}   

df = pd.read_csv(f"{DATA}/labels.csv")     
short = lambda s: s.replace("ableton_live_", "AL ").replace("garageband_", "GB ").replace("_", " ")
print(len(df), "clips,", df.instrument.nunique(), "instruments")
df.head()

7398 clips, 16 instruments
<function <lambda> at 0x10ed0b100>


---
## Part 1: Build chroma

**What chroma is.** For each clip, 12 numbers: how much energy sits on C, C♯, …, B, with all octaves added together.

**How (three steps):**
1. Cut the clip into overlapping 0.5 s frames and take the power spectrum of each (FFT, squared). Average the frames.
2. Give every FFT frequency bin to its nearest semitone. Example: 261.6 Hz → MIDI 60 → C.
3. Fold octaves: MIDI 60, 72, 84 are all C, so `PITCH_CLASSES = midi % 12`. Add up the power in each of the 12 boxes.

Then divide by the total, so each chroma row **sums to 1** (the fingerprint code assumes this).

This is the roadmap's Stage 3 code. It also makes **bass chroma** (same thing, 40–250 Hz only), which we save for the main classifier but don't use in E1.

In [ ]:
import numpy as np
import librosa   # pip install librosa

def features(clip, sr=SR):
    y = clip[int(0.05 * sr):].astype(np.float32)            # skip the pluck noise, same as before
    main = librosa.feature.chroma_cqt(y=y, sr=sr, fmin=librosa.note_to_hz("A1"),
                                      n_octaves=6, norm=None).mean(axis=1)   # A1 to A7
    bass = librosa.feature.chroma_cqt(y=y, sr=sr, fmin=librosa.note_to_hz("E1"),
                                      n_octaves=2, norm=None).mean(axis=1)   # E1 to E3
    total = main.sum() + 1e-12
    return np.concatenate([main / total, bass / total])     # same 24 numbers, C first



**Run it on all 7,398 clips** and save `features.npy` (7,398 × 24). Takes well under a minute.
If the file already exists, this just loads it. Delete it to rebuild (for example after changing `ATTACK_S`).

In [6]:
FEAT_PATH = f"{DATA}/features.npy"
X_audio = np.load("data/processed/clips_16k.npy")

if os.path.exists(FEAT_PATH):
    F = np.load(FEAT_PATH)
    print("loaded existing", FEAT_PATH, F.shape)
else:
    clips = np.load(f"{DATA}/clips_16k.npy", mmap_mode="r")   # read lazily from disk instead of loading 470 MB
    t0 = time.time()
    F = np.array([features(c) for c in clips])
    np.save(FEAT_PATH, F)
    print(f"built {F.shape} in {time.time() - t0:.0f} s -> {FEAT_PATH}")

chroma = F[:, :12]      # E1 uses only the 12 chroma numbers

loaded existing data/processed/features.npy (7398, 24)


**Sanity checks.**
- Every chroma row should sum to 1, and no clip should be silent.
- Our chroma should correlate highly (above about 0.9) with librosa's. It won't match exactly, because librosa weights octaves differently.

In [ ]:
import librosa
print("shape:", F.shape)
print("every chroma row sums to 1:", np.allclose(chroma.sum(1), 1))
print("silent clips:", int((F.sum(1) == 0).sum()))

clips = np.load(f"{DATA}/clips_16k.npy", mmap_mode="r")
rng = np.random.default_rng(0)
for i in rng.choice(len(df), 5, replace=False):
    clip = clips[i].astype(np.float32)
    ref = librosa.feature.chroma_stft(y=clip[int(ATTACK_S * SR):], sr=SR, n_fft=N_FFT,
                                      hop_length=HOP, tuning=0).mean(axis=1)
    print(f"{df.harte[i]:12s} on {short(df.instrument[i]):32s} corr with librosa = {np.corrcoef(chroma[i], ref)[0, 1]:.3f}")

**The picture that motivates everything (roadmap Fig 3).** Rotate every clip so its root is C, then average by chord type.

Blue bars are the chord's own notes. **Red bars are energy on notes that aren't in the chord: that's leakage from overtones.**
Look at B in C major (E's fifth and G's major third both land there), or G in C power (only C and G are played, but C's 3rd harmonic is a G too).

In [ ]:
roots = df.root.to_numpy()
shifted = np.array([np.roll(c, -r) for c, r in zip(chroma, roots)])   # shift so the root sits in box 0 (C)

fig, axes = plt.subplots(1, 7, figsize=(19, 2.8), sharey=True)
for ax, q in zip(axes, QUALITIES):
    m = shifted[(df.quality == q).to_numpy()].mean(0)
    ax.bar(range(12), m, color=["tab:blue" if k in QUALITIES[q] else "tab:red" for k in range(12)])
    ax.set_title(f"C {q}"); ax.set_xticks(range(12), PITCH_CLASSES, fontsize=7)
fig.suptitle("Average chroma per chord type, transposed to C   (blue = chord notes, red = leakage)", y=1.05)
plt.show()

---
## Part 2: The fingerprint code, tested on a fake instrument

**The fingerprint *h*:** 12 numbers, the chroma of *one note* on this instrument, measured from that note.
`h[0]` = the note itself (and its octaves), `h[7]` = a fifth up, `h[4]` = a major third up, `h[10]` ≈ a minor seventh up.

**The model:** a chord's chroma = the average of *h* shifted to each of its notes.

The four functions below are the roadmap's E1 code, unchanged:
- `notes_vector`: which pitch classes a chord contains (1 = in the chord).
- `mix_matrix`: builds a 12×12 matrix **M** so that `M @ h` = the model's chroma for that chord. Row *k* of M says which entries of *h* land in box *k*.
- `fit_fingerprint`: stacks the equations from many clips and solves for *h* with NNLS (least squares, no negative values allowed).
- `fit_r2`: how much of the real chroma's variation the fitted *h* explains (1 = all of it).

In [ ]:
CHORDS = [(r, q) for r in range(12) for q in QUALITIES]       # the 84 classes

def notes_vector(root, quality):
    x = np.zeros(12); x[[(root + i) % 12 for i in QUALITIES[quality]]] = 1
    return x

def mix_matrix(x):
    # column d = the chord's notes shifted up d semitones; divide by the number of notes (average)
    return np.stack([np.roll(x, d) for d in range(12)], axis=1) / x.sum()

def fit_fingerprint(chromas, xs):
    # chromas: n x 12 (each row sums to 1); xs: n x 12 note vectors -> h, 12 numbers summing to 1
    A = np.vstack([mix_matrix(x) for x in xs])      # (12n) x 12: one equation per box per clip
    h, _ = nnls(A, np.concatenate(chromas))         # unknowns = the 12 values of h
    return h / h.sum()

def model_chroma(x, h):
    return mix_matrix(x) @ h

def fit_r2(chromas, xs, h):
    pred = np.array([model_chroma(x, h) for x in xs])
    return 1 - ((chromas - pred) ** 2).sum() / ((chromas - chromas.mean(0)) ** 2).sum()

**A tiny example to see what `mix_matrix` does.** C major (C, E, G) on a toy instrument where 70 % of a note's energy stays on the note, 20 % goes a fifth up and 10 % a major third up.

Row 11 of M (the B box) should have 1/3 in columns 4, 7 and 11. In words: **B box = (h[11] + h[7] + h[4]) / 3**, because B is 11 semitones above C, 7 above E and 4 above G.

In [ ]:
x = notes_vector(0, "maj")                     # C major
print("notes vector:", x.astype(int), " (1s at C, E, G)")
M = mix_matrix(x)
print("row 11 of M (B box):", M[11].round(3))

h_toy = np.zeros(12); h_toy[[0, 7, 4]] = [0.7, 0.2, 0.1]
print("\nModel chroma for C major on the toy instrument:")
print(pd.Series(model_chroma(x, h_toy), index=PITCH_CLASSES).round(3).to_string())
print("-> B gets 0.1 although it isn't in the chord: E's fifth (0.2/3) + G's major third (0.1/3)")

**Making a fake instrument.** An ideal tone has harmonics at 1×, 2×, 3×, … the note's frequency. Harmonic *k* lands `round(12·log₂k) mod 12` semitones above the note:

| harmonic *k* | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| lands on | note | note | 5th (7) | note | M3 (4) | 5th (7) | ≈m7 (10) | note |

We give harmonic *k* the amplitude *k*^−α. **Small α = loud overtones = bright; large α = dull.** Chroma adds up *power* (amplitude²), so each harmonic contributes (*k*^−α)².
Folding these into 12 boxes gives *h*. (This is also the start of E3's simulator, but here it's just a source of known answers.)

In [ ]:
def fake_fingerprint(alpha, n_harmonics=20):
    # h for an ideal tone whose harmonic k has amplitude k**-alpha (low alpha = bright)
    k = np.arange(1, n_harmonics + 1)
    interval = np.round(12 * np.log2(k)).astype(int) % 12     # semitones above the note, octaves folded
    energy = (k ** -float(alpha)) ** 2                         # power = amplitude squared
    h = np.bincount(interval, weights=energy, minlength=12)
    return h / h.sum()

print(pd.DataFrame({f"alpha={a}": fake_fingerprint(a) for a in [0.5, 1.0, 2.0]},
                   index=[f"h[{k}]" for k in range(12)]).round(3).T.to_string())

**Recovery test.** Make chroma for a realistic set of chords from a fingerprint we chose (α = 1), then ask `fit_fingerprint` to find *h* without telling it.

- **No noise:** the error should be tiny (~1e-6 or smaller). If not, there's a bug.
- **With noise:** the error grows but should stay small with a few hundred clips, and R² should drop below 1.

We reuse the list of chords of one real instrument (only the labels, not its audio), so the mix of chord types is realistic.

In [ ]:
xs_all = np.array([notes_vector(r, q) for r, q in zip(df.root, df.quality)])   # note vector of every clip
m0 = (df.instrument == df.instrument.unique()[0]).to_numpy()
xs_test = xs_all[m0]                                       # 273 chords' labels

h_true = fake_fingerprint(alpha=1.0)
clean = np.array([model_chroma(x, h_true) for x in xs_test])   # perfect fake chroma

def add_noise(C, sd, rng):
    noisy = np.clip(C + rng.normal(0, sd, C.shape), 0, None)   # random wobble; energy can't go negative
    return noisy / noisy.sum(1, keepdims=True)                 # re-scale so each row sums to 1 again

rng = np.random.default_rng(0)
for sd in [0, 0.01, 0.03]:
    C = clean if sd == 0 else add_noise(clean, sd, rng)
    h_fit = fit_fingerprint(C, xs_test)
    print(f"noise sd {sd:<5}: largest error in any h[k] = {np.abs(h_fit - h_true).max():.1e},  R² = {fit_r2(C, xs_test, h_fit):.3f}")

**How many labelled clips does a fit need?** (This matters for E2, which fits from only 3.)

For each number of clips, draw 200 random subsets, fit *h*, and measure the error: the total of |fitted − true| over the 12 boxes (0 = perfect, 2 = completely wrong).
The red line is how different a bright (α = 0.7) and a dull (α = 2) fingerprint are. **If the 3-clip error is well below the red line, 3 clips are enough to tell instruments apart.**

`NOISE` is a guess for now. Part 3 prints how noisy real chroma actually is, so we can come back and set it properly.

In [ ]:
NOISE = 0.02
sizes = [1, 2, 3, 5, 10, 30, 100, len(xs_test)]
errs = []
for n in sizes:
    e = []
    for _ in range(200):
        idx = rng.choice(len(xs_test), n, replace=False)
        h_fit = fit_fingerprint(add_noise(clean[idx], NOISE, rng), xs_test[idx])
        e.append(np.abs(h_fit - h_true).sum())
    errs.append(e)

gap = np.abs(fake_fingerprint(0.7) - fake_fingerprint(2.0)).sum()
plt.figure(figsize=(7, 3.5))
plt.boxplot(errs); plt.xticks(range(1, len(sizes) + 1), sizes)
plt.axhline(gap, ls="--", c="r", label=f"bright vs dull difference ({gap:.2f})")
plt.xlabel("labelled clips used for the fit"); plt.ylabel("error in fitted h"); plt.legend(); plt.show()
print("median error with 3 clips:", round(float(np.median(errs[2])), 3))

---
## Part 3: Fit *h* for the real instruments

For each instrument: fit *h* from all its clips, then measure **R²** (how much of its chroma one fingerprint explains) and **resid_sd** (the typical size of what's left over, i.e. the real "noise").

**Data hygiene.** We should use only the 13 development instruments and keep the 3 test instruments (from `splits.json`) out.
`splits.json` doesn't exist yet, so for now this fits all 16 and says so. E1 only describes instruments and tunes nothing, so this is low-risk, but pick the test instruments before E2.

In [ ]:
SPLITS = f"{DATA}/splits.json"
TEST = json.load(open(SPLITS))["test"] if os.path.exists(SPLITS) else []
if not TEST:
    print("No splits.json yet: fitting all 16 instruments for now.\n")
dev = [i for i in df.instrument.unique() if i not in TEST]

rows, H = [], {}
for inst in dev:
    m = (df.instrument == inst).to_numpy()
    h = fit_fingerprint(chroma[m], xs_all[m])
    pred = np.array([model_chroma(x, h) for x in xs_all[m]])
    H[short(inst)] = h
    rows.append(dict(instrument=short(inst), clips=int(m.sum()),
                     R2=fit_r2(chroma[m], xs_all[m], h), resid_sd=(chroma[m] - pred).std(),
                     h0=h[0], h7=h[7], h4=h[4], h10=h[10]))

table = pd.DataFrame(rows).set_index("instrument").sort_values("h0")
print(f"(Part 2 used NOISE = {NOISE}; real resid_sd ranges {table.resid_sd.min():.3f}-{table.resid_sd.max():.3f})")
table.round(3)

**The fingerprints as a heatmap (draft of Fig 8).** One row per instrument, one column per interval above the note.
Rows are sorted by `h[0]`: instruments at the top keep the least energy on the note itself, so they leak the most.
The colour scale is square-rooted so the small leakage values are visible next to the big `h[0]`.

Next step (H1 check): compare these with each instrument's brightness.

In [ ]:
from matplotlib.colors import PowerNorm
names = table.index
Hm = np.array([H[n] for n in names])
fig, ax = plt.subplots(figsize=(10, 0.45 * len(names) + 1.5))
im = ax.imshow(Hm, cmap="magma", aspect="auto", norm=PowerNorm(0.5))
for i in range(Hm.shape[0]):
    for k in range(12):
        ax.text(k, i, f"{Hm[i, k]:.2f}", ha="center", va="center", fontsize=7,
                color="black" if Hm[i, k] > 0.3 else "white")
ax.set_xticks(range(12), ["0\nnote", "1", "2", "3\nm3", "4\nM3", "5", "6", "7\n5th", "8", "9", "10\nm7", "11\nM7"])
ax.set_yticks(range(len(names)), names, fontsize=8)
ax.set_xlabel("semitones above the note"); fig.colorbar(im, label="share of one note's energy")
ax.set_title("Fitted fingerprint h per instrument"); plt.tight_layout(); plt.show()